In [ ]:
from Classes.Grid import Grid
from Classes.ScalarCoeffs import ScalarCoeffs
from Classes.BoundaryConditions import BoundaryLocation, DirichletBc, NeumannBc, RobinBc
from Classes.Models import DiffusionModel, SurfaceConvectionModel, FirstOrderTransientModel, SecondOrderTransientModel, UpwindAdvectionModel, CDSAdvectionModel, QUICKAdvectionModel
from Classes.LinearSolver import solve

import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt

class AdvectingVelocityModel:
    """Class defining an advecting velocity model"""

    def __init__(self, grid, dhat, Uhe, P, U, coeffs):
        """Constructor"""
        self._grid = grid
        self._dhat = dhat
        self._Uhe = Uhe
        self._P = P
        self._U = U
        self._coeffs = coeffs

    def update(self):
        """Function to update the advecting velocity array"""

        # Calculate the pressure gradients across the faces
        gradPw = (self._P[1:-1]-self._P[0:-2])/self._grid.dx_WP
        gradPe = (self._P[2:]-self._P[1:-1])/self._grid.dx_PE
        
        # Calculate the cell pressure gradients
        gradP = 0.5*(gradPw + gradPe)
          
        # Calculate damping coefficient, dhat
        Ve = 0.5*(self._grid.vol[0:-1] + self._grid.vol[1:])
        ae = 0.5*(self._coeffs.aP[0:-1] + self._coeffs.aP[1:])
        self._dhat[1:-1] = Ve/ae

        # Update the advecting velocity
        self._Uhe[0] = self._U[0]
        self._Uhe[1:-1] = 0.5*(self._U[1:-2] + self._U[2:-1]) - self._dhat[1:-1]*(gradPe[:-1] - 0.5*(gradP[:-1] + gradP[1:]))
        self._Uhe[-1] = self._U[-1]
        
class PressureForceModel:
    """Class defining a pressure force model"""

    def __init__(self, grid, P, west_bc, east_bc):
        """Constructor"""
        self._grid = grid
        self._P = P
        self._west_bc = west_bc
        self._east_bc = east_bc

    def add(self, coeffs):
        """Function to add diffusion terms to coefficient arrays"""

        # Calculate the pressure force
        gradPw = (self._P[1:-1]-self._P[0:-2])/self._grid.dx_WP
        gradPe = (self._P[2:]-self._P[1:-1])/self._grid.dx_PE
        force = 0.5*(gradPw + gradPe)*self._grid.vol
          
        # Calculate the linearization coefficients
        coeffW = - 0.5*self._grid.vol/self._grid.dx_WP
        coeffE = 0.5*self._grid.vol/self._grid.dx_PE
        coeffP = - coeffW - coeffE

        # Modify the linearization coefficients on the boundaries
        coeffP[0] += coeffW[0]*self._west_bc.coeff()
        coeffP[-1] += coeffE[-1]*self._east_bc.coeff()

        # Zero the boundary coefficients that are not used
        coeffW[0] = 0.0
        coeffE[-1] = 0.0

        # Add to coefficient arrays
        coeffs.accumulate_aP(coeffP)
        coeffs.accumulate_aW(coeffW)
        coeffs.accumulate_aE(coeffE)
        coeffs.accumulate_rP(force)

        # Return the modified coefficient array
        return coeffs
        
class MassConservationEquation:
    """Class defining a mass conservation equation"""

    def __init__(self, grid, U, P, dhat, Uhe, rho, 
                 P_west_bc, P_east_bc, U_west_bc, U_east_bc):
        """Constructor"""
        self._grid = grid
        self._U = U
        self._P = P
        self._dhat = dhat
        self._Uhe = Uhe
        self._rho = rho
        self._P_west_bc = P_west_bc
        self._P_east_bc = P_east_bc
        self._U_west_bc = U_west_bc
        self._U_east_bc = U_east_bc

    def add(self, PP_coeffs, PU_coeffs):
        """Function to add diffusion terms to coefficient arrays"""

        # Calculate the mass imbalance, based on advecting velocities
        imbalance = self._rho*self._grid.Ae*self._Uhe[1:] - self._rho*self._grid.Aw*self._Uhe[:-1]
              
        # Calculate the linearization coefficients on pressure
        PP_coeffW = np.concatenate((np.array([0]), -self._rho*self._grid.Aw[1:]*self._dhat[1:-1]/self._grid.dx_WP[1:]))
        PP_coeffE = np.concatenate((-self._rho*self._grid.Ae[:-1]*self._dhat[1:-1]/self._grid.dx_PE[:-1], np.array([0])))
        PP_coeffP = - PP_coeffW - PP_coeffE
        
        # Calculate the linearization coefficients on velocity
        PU_coeffW = np.concatenate((np.array([-self._rho*self._grid.Aw[0]]), -0.5*self._rho*self._grid.Aw[1:]))
        PU_coeffE = np.concatenate((0.5*self._rho*self._grid.Ae[:-1], np.array([self._rho*self._grid.Ae[-1]])))
        PU_coeffP = np.concatenate((np.array([0]), PU_coeffW[1:])) + np.concatenate((PU_coeffE[:-1], np.array([0])))

        # Modify the linearization coefficients on the boundaries 
        # (velocity only, since pressure is already zero)
        PU_coeffP[0] += PU_coeffW[0]*self._U_west_bc.coeff()
        PU_coeffP[-1] += PU_coeffE[-1]*self._U_east_bc.coeff()

        # Zero the boundary coefficients that are not used
        PU_coeffW[0] = 0.0
        PU_coeffE[-1] = 0.0

        # Add to coefficient arrays
        PP_coeffs.accumulate_aP(PP_coeffP)
        PP_coeffs.accumulate_aW(PP_coeffW)
        PP_coeffs.accumulate_aE(PP_coeffE)
        PP_coeffs.accumulate_rP(imbalance)
        PU_coeffs.accumulate_aP(PU_coeffP)
        PU_coeffs.accumulate_aW(PU_coeffW)
        PU_coeffs.accumulate_aE(PU_coeffE)

        # Return the modified coefficient arrays
        return PP_coeffs, PU_coeffs

class ExtrapolatedBc:
    """Class defining an extrapolated boundary condition"""

    def __init__(self, phi, grid, loc):
        """Constructor
            phi ........ field variable array
            grid ....... grid
            loc ........ boundary location
        """
        self._phi = phi
        self._grid = grid
        self._loc = loc

    def value(self):
        """Return the boundary condition value"""
        if self._loc is BoundaryLocation.WEST:
            return (self._phi[1] - ((self._phi[2] - self._phi[1]) / self._grid.dx_PE[0]) * self._grid.dx_WP[0])

        elif self._loc is BoundaryLocation.EAST:
            return (self._phi[-2] + ((self._phi[-2] - self._phi[-3]) / self._grid.dx_WP[-1]) * self._grid.dx_PE[-1])
        else:
            raise ValueError("Unknown boundary location")

    def coeff(self):
        """Return the linearization coefficient"""
        if self._loc is BoundaryLocation.WEST:
            return 1 + self._grid.dx_WP[0] / self._grid.dx_PE[0]
        elif self._loc is BoundaryLocation.EAST:
            return  1 + self._grid.dx_PE[-1] / self._grid.dx_WP[-1]
        else:
            raise ValueError("Unknown boundary location")

    def apply(self):
        """Applies the boundary condition in the referenced field variable array"""
        if self._loc is BoundaryLocation.WEST:
            pass # Fill in expression below
            self._phi[0] =  (self._phi[1] - ((self._phi[2] - self._phi[1]) / self._grid.dx_PE[0]) * self._grid.dx_WP[0])
        elif self._loc is BoundaryLocation.EAST:
            pass # Fill in expression below
            self._phi[-1] = (self._phi[-2] + ((self._phi[-2] - self._phi[-3]) / self._grid.dx_WP[-1]) * self._grid.dx_PE[-1])
        else:
            raise ValueError("Unknown boundary location")

# Assignment 4 - Solution of Mass and Momentum Equations

Solve the following problems and explain your results.

Solve all problems using water as the fluid with $\rho=1000$ [kg/m$^3$], $\mu=1 \times 10^{-3}$ [kg/m$\cdot$s].  Problems 1-2 should be solved for a 4 [m] long, 0.02 $\times$ 0.02 [m] cross-section duct discretized using 10 equal-length control-volumes.  Also, for simplicity, use UDS as your advection scheme for problems 1-2.  Higher-order advection schemes are considered in question 3.

Heat transfer is not considered in these problems, so the energy equation does not need to be solved.

## Problem 1

For $\tau_w=0$, set $u = \hat{u} = 10$ [m/s] and $p = 0$ [Pa] everywhere as initial conditions.  These are the exact solutions to the mass and momentum equations for constant duct area.  Do one time step (using any time step size) and make sure that your code accepts this as the exact solution.  Repeat this problem with $u = \hat{u} = -10$ [m/s] to ensure that your code has no directional dependence.  Describe the boundary conditions used for the two cases, including the values of $\alpha_e$.

## u =10

In [ ]:
# Define the grid
lx = 4.0
ly = 0.02
lz = 0.02
ncv = 10
grid = Grid(lx, ly, lz, ncv)

# Set the timestep information
nTime = 1
dt = 1e9
time = 0

# Set the maximum number of iterations and convergence criterion
maxIter = 100
converged = 1e-6

# Define thermophysical properties
rho = 1000
mu = 1e-3

# Define the coefficients
PU_coeffs = ScalarCoeffs(grid.ncv)
PP_coeffs = ScalarCoeffs(grid.ncv)
UP_coeffs = ScalarCoeffs(grid.ncv)
UU_coeffs = ScalarCoeffs(grid.ncv)

# Initial conditions
U0 = 10
P0 = 0

# Initialize field variable arrays
U = U0*np.ones(grid.ncv+2)
P = P0*np.ones(grid.ncv+2)

# Initialize advecting velocity and damping coefficient array
dhat = np.zeros(grid.ncv+1)
Uhe = U0*np.ones(grid.ncv+1)

# Define boundary conditions for velocity
U_west_bc = DirichletBc(U, grid, U0, BoundaryLocation.WEST)
U_east_bc = NeumannBc(U, grid, 0, BoundaryLocation.EAST)

# Define boundary conditions for pressure
#   - Once ExtrapolatedBc is complete, change the boundary condition
P_west_bc = ExtrapolatedBc(P, grid, BoundaryLocation.WEST)
P_east_bc = DirichletBc(P, grid, 0, BoundaryLocation.EAST)

# Apply boundary conditions
U_west_bc.apply()
U_east_bc.apply()
P_west_bc.apply()
P_east_bc.apply()

# Define the transient model
Uold = np.copy(U)
transient = FirstOrderTransientModel(grid, U, Uold, rho, 1, dt)

# Define the diffusion model
diffusion = DiffusionModel(grid, U, mu, U_west_bc, U_east_bc)

# Define the advection model
advection = UpwindAdvectionModel(grid, U, Uhe, rho, 1, U_west_bc, U_east_bc)

# Define the pressure force model
pressure = PressureForceModel(grid, P, P_west_bc, P_east_bc)

# Define advecting velocity model
advecting = AdvectingVelocityModel(grid, dhat, Uhe, P, U, UU_coeffs)

# Define conservation of mass equation
mass = MassConservationEquation(grid, U, P, dhat, Uhe, rho, 
                                P_west_bc, P_east_bc, U_west_bc, U_east_bc)

# Loop through all timesteps
for tStep in range(nTime):
    # Update the time information
    time += dt
    
    # Print the timestep information
    print("Timestep = {}; Time = {}".format(tStep, time))
    
    # Store the "old" velocity field
    Uold[:] = U[:]
    
    # Iterate until the solution is converged
    for i in range(maxIter):
        
        # Zero all of the equations
        PP_coeffs.zero()
        PU_coeffs.zero()
        UU_coeffs.zero()
        UP_coeffs.zero()     
        
        # Assemble the momentum equations
        #   Note: do this before mass, because the coeffs are needed to compute advecting velocity
        UU_coeffs = diffusion.add(UU_coeffs)
        UU_coeffs = advection.add(UU_coeffs)
        UU_coeffs = transient.add(UU_coeffs)
        UP_coeffs = pressure.add(UP_coeffs)
        
        # Assemble the mass equations
        advecting.update()
        PP_coeffs, PU_coeffs = mass.add(PP_coeffs, PU_coeffs)

        # Compute residuals and check for convergence
        PmaxResid = norm(PU_coeffs.rP + PP_coeffs.rP, np.inf)
        PavgResid = np.mean(np.absolute(PU_coeffs.rP + PP_coeffs.rP))
        UmaxResid = norm(UU_coeffs.rP + UP_coeffs.rP, np.inf)
        UavgResid = np.mean(np.absolute(UU_coeffs.rP + UP_coeffs.rP))
        print("Iteration = {}.".format(i))
        print("  Mass:     Max. Resid. = {}; Avg. Resid. = {}".format(PmaxResid, PavgResid))
        print("  Momentum: Max. Resid. = {}; Avg. Resid. = {}".format(UmaxResid, UavgResid))
        if PmaxResid < converged and UmaxResid < converged:
            break
    
        # Solve the sparse matrix system
        dP, dU = solve(PP_coeffs, PU_coeffs, UP_coeffs, UU_coeffs)
    
        # Update the solutions 
        P[1:-1] += dP
        U[1:-1] += dU
        
        # Update boundary conditions
        U_west_bc.apply()
        U_east_bc.apply()
        P_west_bc.apply()
        P_east_bc.apply()
        
        # Update the advecting velocities
        advecting.update()


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# Interior cell-center locations
xP = grid.xP

# Full array including boundary/ghost values
# Assumption: west boundary at x = 0, east boundary at x = lx

x_full = grid.xP


# Plot velocity U

plt.figure(figsize=(7, 4))

plt.plot(
    x_full,
    U,
    marker="o",
    linestyle="-",
    label="U"
)

plt.xlabel("x")
plt.ylabel("Velocity, U")
plt.title("Velocity solution")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot pressure P

plt.figure(figsize=(7, 4))

plt.plot(
    x_full,
    P,
    marker="s",
    linestyle="-",
    label="P"
)

plt.xlabel("x")
plt.ylabel("Pressure, P")
plt.title("Pressure solution")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## u =-10

In [ ]:
# Define the grid
lx = 4.0
ly = 0.02
lz = 0.02
ncv = 10
grid = Grid(lx, ly, lz, ncv)

# Set the timestep information
nTime = 1
dt = 1e9
time = 0

# Set the maximum number of iterations and convergence criterion
maxIter = 100
converged = 1e-6

# Define thermophysical properties
rho = 1000
mu = 1e-3

# Define the coefficients
PU_coeffs = ScalarCoeffs(grid.ncv)
PP_coeffs = ScalarCoeffs(grid.ncv)
UP_coeffs = ScalarCoeffs(grid.ncv)
UU_coeffs = ScalarCoeffs(grid.ncv)

# Initial conditions
U0 = -10
P0 = 0

# Initialize field variable arrays
U = U0*np.ones(grid.ncv+2)
P = P0*np.ones(grid.ncv+2)

# Initialize advecting velocity and damping coefficient array
dhat = np.zeros(grid.ncv+1)
Uhe = U0*np.ones(grid.ncv+1)

# Define boundary conditions for velocity
U_west_bc = NeumannBc(U, grid, 0, BoundaryLocation.WEST)
U_east_bc = DirichletBc(U, grid, U0, BoundaryLocation.EAST)


# Define boundary conditions for pressure
#   - Once ExtrapolatedBc is complete, change the boundary condition
P_west_bc = DirichletBc(P, grid, 0, BoundaryLocation.WEST)
P_east_bc = ExtrapolatedBc(P, grid, BoundaryLocation.EAST)

# Apply boundary conditions
U_west_bc.apply()
U_east_bc.apply()
P_west_bc.apply()
P_east_bc.apply()

# Define the transient model
Uold = np.copy(U)
transient = FirstOrderTransientModel(grid, U, Uold, rho, 1, dt)

# Define the diffusion model
diffusion = DiffusionModel(grid, U, mu, U_west_bc, U_east_bc)

# Define the advection model
advection = UpwindAdvectionModel(grid, U, Uhe, rho, 1, U_west_bc, U_east_bc)

# Define the pressure force model
pressure = PressureForceModel(grid, P, P_west_bc, P_east_bc)

# Define advecting velocity model
advecting = AdvectingVelocityModel(grid, dhat, Uhe, P, U, UU_coeffs)

# Define conservation of mass equation
mass = MassConservationEquation(grid, U, P, dhat, Uhe, rho, 
                                P_west_bc, P_east_bc, U_west_bc, U_east_bc)

# Loop through all timesteps
for tStep in range(nTime):
    # Update the time information
    time += dt
    
    # Print the timestep information
    print("Timestep = {}; Time = {}".format(tStep, time))
    
    # Store the "old" velocity field
    Uold[:] = U[:]
    
    # Iterate until the solution is converged
    for i in range(maxIter):
        
        # Zero all of the equations
        PP_coeffs.zero()
        PU_coeffs.zero()
        UU_coeffs.zero()
        UP_coeffs.zero()     
        
        # Assemble the momentum equations
        #   Note: do this before mass, because the coeffs are needed to compute advecting velocity
        UU_coeffs = diffusion.add(UU_coeffs)
        UU_coeffs = advection.add(UU_coeffs)
        UU_coeffs = transient.add(UU_coeffs)
        UP_coeffs = pressure.add(UP_coeffs)
        
        # Assemble the mass equations
        advecting.update()
        PP_coeffs, PU_coeffs = mass.add(PP_coeffs, PU_coeffs)

        # Compute residuals and check for convergence
        PmaxResid = norm(PU_coeffs.rP + PP_coeffs.rP, np.inf)
        PavgResid = np.mean(np.absolute(PU_coeffs.rP + PP_coeffs.rP))
        UmaxResid = norm(UU_coeffs.rP + UP_coeffs.rP, np.inf)
        UavgResid = np.mean(np.absolute(UU_coeffs.rP + UP_coeffs.rP))
        print("Iteration = {}.".format(i))
        print("  Mass:     Max. Resid. = {}; Avg. Resid. = {}".format(PmaxResid, PavgResid))
        print("  Momentum: Max. Resid. = {}; Avg. Resid. = {}".format(UmaxResid, UavgResid))
        if PmaxResid < converged and UmaxResid < converged:
            break
    
        # Solve the sparse matrix system
        dP, dU = solve(PP_coeffs, PU_coeffs, UP_coeffs, UU_coeffs)
    
        # Update the solutions 
        P[1:-1] += dP
        U[1:-1] += dU
        
        # Update boundary conditions
        U_west_bc.apply()
        U_east_bc.apply()
        P_west_bc.apply()
        P_east_bc.apply()
        
        # Update the advecting velocities
        advecting.update()


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np


x_full = grid.xP


# Plot velocity U

plt.figure(figsize=(7, 4))

plt.plot(
    x_full,
    U,
    marker="o",
    linestyle="-",
    label="U"
)

plt.xlabel("x")
plt.ylabel("Velocity, U")
plt.title("Velocity solution")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot pressure P

plt.figure(figsize=(7, 4))

plt.plot(
    x_full,
    P,
    marker="s",
    linestyle="-",
    label="P"
)

plt.xlabel("x")
plt.ylabel("Pressure, P")
plt.title("Pressure solution")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Discussion
For both U0 = 10 and U0 = -10, the exact solution is a uniform velocity field and a uniform zero-pressure field. Since the duct area is constant, the mass flow rate is constant along the domain. Therefore, the continuity equation is satisfied exactly. Also, because the velocity is uniform, the diffusion term is zero, and because the pressure is uniform, the pressure-gradient force is zero. The momentum equation is therefore also satisfied exactly.

For the positive-flow case, the inlet is at the west boundary and the outlet is at the east boundary. A Dirichlet velocity condition U = 10 is applied at the west inlet, with a Neumann zero gradient velocity condition is applied at the east outlet. The pressure is fixed to P = 0 at the east outlet, and the west pressure is extrapolated. Since the flow is from left to right, alphae = 1 is used at the boundaries.

For the reversed-flow case, the inlet and outlet are switched. A Dirichlet velocity condition U = -10 is applied at the east inlet, with a Neumann zero gradient velocity condition is applied at the west outlet. The pressure is fixed to P = 0 at the west outlet, and the east pressure is extrapolated. Since the flow is from right to left, alphae = -1 is used at the boundaries.

## Problem 2

For turbulent flow in a long duct, the wall shear stress can be approximated by:

$$
    \frac{\tau_w}{\frac{1}{2} \rho U^2} = C_f = (1.58 \ln(Re)-3.28)^{-2}
$$

where $Re=\rho D_h U/\mu > 10^4$ and $D_h=4A/P_o$ is the hydraulic diameter of the duct.  Implement the wall shear stress model into your code and  linearize appropriately.  Note that the force on a control-volume can be computed using the above expression as:

$$
    F_u = \tau_w A_o = C_f \frac{1}{2} \rho U^2 A_o
$$

Note that $C_f$ changes very slowly with $U$, so you will only need to linearize the $U^2$ term (not the implicit dependence of $Re$ on $U$).  Impose suitable boundary conditions on the ends of the duct and initialize the problem with $u = \hat{u} = 10$ [m/s] and $p = 0$ [Pa].  Check that the pressure is exactly correct after emerging from enough iterations by comparing your result with the exact solution calculated from the above expressions. Why is the result not correct after the first iteration?

In [ ]:
def exact_pressure_wall_shear(x, U0, rho, mu, A, Po, L):
    Dh = 4.0 * A / Po

    Re = rho * Dh * abs(U0) / mu
    Cf = (1.58 * np.log(Re) - 3.28) ** (-2)

    dpdx_mag = 0.5 * Cf * rho * U0**2 * Po / A

    if U0 > 0:
        # Flow west -> east, outlet pressure fixed at x = L
        P_exact = dpdx_mag * (L - x)
    else:
        # Flow east -> west, outlet pressure fixed at x = 0
        P_exact = dpdx_mag * x

    return P_exact, Cf, Re
    
class WallShearStressModel:
    """Wall shear stress source model for turbulent duct flow."""

    def __init__(self, grid, U, rho, mu, Dh, Ao):
        """
        grid ...... Grid object
        U ......... velocity array, including ghost cells
        rho ....... density
        mu ........ dynamic viscosity
        Dh ........ hydraulic diameter
        Ao ........ wall area per control volume, array of length ncv
        """
        self._grid = grid
        self._U = U
        self._rho = rho
        self._mu = mu
        self._Dh = Dh
        self._Ao = Ao

    def add(self, coeffs):
        """Add wall shear stress contribution to momentum equation."""

        # Interior cell velocities
        UP = self._U[1:-1]

        # Reynolds number based on local velocity magnitude
        Re = self._rho * self._Dh * np.abs(UP) / self._mu

        # Turbulent friction coefficient
        Cf = (1.58 * np.log(Re) - 3.28) ** (-2)

        # Wall shear force:
        Fu = 0.5 * Cf * self._rho * UP * np.abs(UP) * self._Ao

        # Linearization:
        coeffP = Cf * self._rho  * self._Ao

        coeffs.accumulate_aP(coeffP)
        coeffs.accumulate_rP(Fu)

        return coeffs

In [ ]:
# Define Geometry
lx = 4.0
ly = 0.02
lz = 0.02
ncv = 10

A = ly * lz
Po = 2.0 * (ly + lz)
Dh = 4.0 * A / Po

dx = lx / ncv
Ao = Po * dx * np.ones(ncv)

# Set the timestep information
nTime = 1
dt = 1e9
time = 0

# Set the maximum number of iterations and convergence criterion
maxIter = 100
converged = 1e-6

# Define thermophysical properties
rho = 1000
mu = 1e-3

# Define the coefficients
PU_coeffs = ScalarCoeffs(grid.ncv)
PP_coeffs = ScalarCoeffs(grid.ncv)
UP_coeffs = ScalarCoeffs(grid.ncv)
UU_coeffs = ScalarCoeffs(grid.ncv)

# Initial conditions
U0 = 10
P0 = 0

# Initialize field variable arrays
U = U0*np.ones(grid.ncv+2)
P = P0*np.ones(grid.ncv+2)

# Initialize advecting velocity and damping coefficient array
dhat = np.zeros(grid.ncv+1)
Uhe = U0*np.ones(grid.ncv+1)

# Define boundary conditions for velocity
U_west_bc = DirichletBc(U, grid, 10, BoundaryLocation.WEST)
U_east_bc = NeumannBc(U, grid, 0, BoundaryLocation.EAST)

# Define boundary conditions for pressure
#   - Once ExtrapolatedBc is complete, change the boundary condition
P_west_bc = ExtrapolatedBc(P, grid, BoundaryLocation.WEST)
P_east_bc = DirichletBc(P, grid, 0, BoundaryLocation.EAST)

# Apply boundary conditions
U_west_bc.apply()
U_east_bc.apply()
P_west_bc.apply()
P_east_bc.apply()

# Define the transient model
Uold = np.copy(U)
transient = FirstOrderTransientModel(grid, U, Uold, rho, 1, dt)

# Define the diffusion model
diffusion = DiffusionModel(grid, U, mu, U_west_bc, U_east_bc)

# Define the advection model
advection = UpwindAdvectionModel(grid, U, Uhe, rho, 1, U_west_bc, U_east_bc)

# Define the pressure force model
pressure = PressureForceModel(grid, P, P_west_bc, P_east_bc)

# Define advecting velocity model
advecting = AdvectingVelocityModel(grid, dhat, Uhe, P, U, UU_coeffs)

# Define conservation of mass equation
mass = MassConservationEquation(grid, U, P, dhat, Uhe, rho, 
                                P_west_bc, P_east_bc, U_west_bc, U_east_bc)
# Define wall shear
wall_shear = WallShearStressModel(grid, U, rho, mu, Dh, Ao)

# Loop through all timesteps
for tStep in range(nTime):
    # Update the time information
    time += dt
    
    # Print the timestep information
    print("Timestep = {}; Time = {}".format(tStep, time))
    
    # Store the "old" velocity field
    Uold[:] = U[:]
    
    # Iterate until the solution is converged
    for i in range(maxIter):
        
        # Zero all of the equations
        PP_coeffs.zero()
        PU_coeffs.zero()
        UU_coeffs.zero()
        UP_coeffs.zero()     
        
        # Assemble the momentum equations
        #   Note: do this before mass, because the coeffs are needed to compute advecting velocity
        UU_coeffs = diffusion.add(UU_coeffs)
        UU_coeffs = advection.add(UU_coeffs)
        UU_coeffs = transient.add(UU_coeffs)
        UU_coeffs = wall_shear.add(UU_coeffs)
        UP_coeffs = pressure.add(UP_coeffs)
        
        # Assemble the mass equations
        advecting.update()
        PP_coeffs, PU_coeffs = mass.add(PP_coeffs, PU_coeffs)

        # Compute residuals and check for convergence
        PmaxResid = norm(PU_coeffs.rP + PP_coeffs.rP, np.inf)
        PavgResid = np.mean(np.absolute(PU_coeffs.rP + PP_coeffs.rP))
        UmaxResid = norm(UU_coeffs.rP + UP_coeffs.rP, np.inf)
        UavgResid = np.mean(np.absolute(UU_coeffs.rP + UP_coeffs.rP))
        print("Iteration = {}.".format(i))
        print("  Mass:     Max. Resid. = {}; Avg. Resid. = {}".format(PmaxResid, PavgResid))
        print("  Momentum: Max. Resid. = {}; Avg. Resid. = {}".format(UmaxResid, UavgResid))
        if PmaxResid < converged and UmaxResid < converged:
            break
    
        # Solve the sparse matrix system
        dP, dU = solve(PP_coeffs, PU_coeffs, UP_coeffs, UU_coeffs)
    
        # Update the solutions 
        P[1:-1] += dP
        U[1:-1] += dU
        
        # Update boundary conditions
        U_west_bc.apply()
        U_east_bc.apply()
        P_west_bc.apply()
        P_east_bc.apply()
        
        # Update the advecting velocities
        advecting.update()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

x_an = np.linspace(0.0, lx, 500)

P_an, Cf, Re = exact_pressure_wall_shear(
    x_an,
    U0,
    rho,
    mu,
    A,
    Po,
    lx
)

print("Re =", Re)
print("Cf =", Cf)

plt.figure(figsize=(7, 4))
plt.plot(grid.xP, P, "o", label="Numerical P")
plt.plot(x_an, P_an, "-", label="Exact P")
plt.xlabel("x")
plt.ylabel("Pressure, P")
plt.title(f"Pressure solution, U0 = {U0} m/s")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(grid.xP, U, "o-", label="Numerical U")
plt.xlabel("x")
plt.ylabel("Velocity, U")
plt.ylim(9.5, 10.5)
plt.title(f"Velocity solution, U0 = {U0} m/s")
plt.grid(True)
plt.legend()
plt.show()

## Problem 3

In this problem, we explore the flow in a frictionless converging-diverging circular duct.  We will now consider the utility of second-order advection schemes to explore errors associated with UDS.  Implement the CDS and QUICK schemes into the momentum equation.  You can essentially use what you created in the previous assignment for this task.

![Duct](Figures/4-Duct.png)

The duct is defined by:
$$
    r=2H_t + H_t cos \left( 2\pi \frac{x}{L} \right)
$$

where $L=1$ [m], $H_t=0.01$ [m].  You will have to modify the `Grid` class in order to solve this problem. Keep the general structure of the class, but modify the calculation of the areas and volumes appropriately. The inlet velocity should be imposed as $u=2$ [m/s].  To eliminate friction in the duct, be sure to turn off the wall friction terms from the previous problem.  Solve the problem using 8, 16, 32 and 64 equal-length control-volumes and calculate the loss in dynamic head from each converged solution.  The dynamic head loss is given as:

$$
    C_D= \frac{P_{in} - P_{out}}{\frac{1}{2} \rho U_{in}^2}
$$

Compare your solutions from UDS with those from the second-order schemes and quantify the convergence characteristics of each.  Plot the velocities and pressures versus $x$ for enough of the cases to visualize the results properly.  What should $C_D$ become for this problem?

In [ ]:
import numpy as np

class Grid_variable_circular:
    """One-dimensional variable-area circular duct grid"""

    def __init__(self, lx, Ht, lz, ncv):
        """
        lx  .... duct length [m]
        Ht  .... length scale in radius equation [m]
        lz  .... unused, kept for compatibility
        ncv .... number of control volumes

        Radius:
            r(x) = 2Ht + Ht*cos(2*pi*x/L)
        """

        self._ncv = ncv
        self._lx = lx
        self._Ht = Ht

        dx = lx / float(ncv)
        self._dx = dx

        # Face locations
        self._xf = np.array([i * dx for i in range(ncv + 1)])

        # Cell-center locations with boundary points included
        self._xP = np.array(
            [self._xf[0]]
            + [0.5 * (self._xf[i] + self._xf[i + 1]) for i in range(ncv)]
            + [self._xf[-1]]
        )

        # Radius at faces and cell centers
        self._rf = self.radius(self._xf)
        self._rP = self.radius(self._xP)

        # Face areas
        self._Af = np.pi * self._rf**2

        # Cell volumes: exact integral of pi*r(x)^2 dx over each CV
        self._vol = np.zeros(ncv)
        for i in range(ncv):
            self._vol[i] = self.integral_area(self._xf[i], self._xf[i + 1])

        # Approximate wall area for each CV
        # Not important here because wall friction is turned off.
        self._Ao = np.zeros(ncv)
        for i in range(ncv):
            rW = self._rf[i]
            rE = self._rf[i + 1]
            slant = np.sqrt(dx**2 + (rE - rW)**2)
            self._Ao[i] = np.pi * (rW + rE) * slant

    def radius(self, x):
        """Radius distribution"""
        return 2.0 * self._Ht + self._Ht * np.cos(2.0 * np.pi * x / self._lx)

    def area(self, x):
        """Cross-sectional area"""
        r = self.radius(x)
        return np.pi * r**2

    def integral_area(self, x1, x2):
        """
        Exact integral of A(x) = pi*r(x)^2 from x1 to x2.
        r = Ht*(2 + cos(ax)), a = 2*pi/L
        """
        Ht = self._Ht
        L = self._lx
        a = 2.0 * np.pi / L

        def F(x):
            return np.pi * Ht**2 * (
                4.5 * x
                + (4.0 / a) * np.sin(a * x)
                + (1.0 / (4.0 * a)) * np.sin(2.0 * a * x)
            )

        return F(x2) - F(x1)

    @property
    def ncv(self):
        return self._ncv

    @property
    def xf(self):
        return self._xf

    @property
    def xP(self):
        return self._xP

    @property
    def dx_WP(self):
        return self.xP[1:-1] - self.xP[0:-2]

    @property
    def dx_PE(self):
        return self.xP[2:] - self.xP[1:-1]

    @property
    def Af(self):
        return self._Af

    @property
    def Aw(self):
        return self._Af[0:-1]

    @property
    def Ae(self):
        return self._Af[1:]

    @property
    def Ao(self):
        return self._Ao

    @property
    def vol(self):
        return self._vol

    @property
    def rf(self):
        return self._rf

    @property
    def rP(self):
        return self._rP

In [ ]:
def run_variable_duct_case(ncv, advection_class):

    # Grid 
    lx = 1.0
    Ht = 0.01
    lz = 0.0

    grid = Grid_variable_circular(lx, Ht, lz, ncv)

    rho = 1000.0


    # Time and iteration settings
    nTime = 1
    dt = 1e9
    time = 0.0

    maxIter = 500
    converged = 1e-8


    PU_coeffs = ScalarCoeffs(grid.ncv)
    PP_coeffs = ScalarCoeffs(grid.ncv)
    UP_coeffs = ScalarCoeffs(grid.ncv)
    UU_coeffs = ScalarCoeffs(grid.ncv)


    U0 = 2
    Pout = 0

    U = U0 * np.ones(grid.ncv + 2)
    P = Pout * np.ones(grid.ncv + 2)

    dhat = np.zeros(grid.ncv + 1)
    Uhe = Uin * np.ones(grid.ncv + 1)

    # Boundary conditions
    U_west_bc = DirichletBc(U, grid, Uin, BoundaryLocation.WEST)
    U_east_bc = NeumannBc(U, grid, 0.0, BoundaryLocation.EAST)

    P_west_bc = ExtrapolatedBc(P, grid, BoundaryLocation.WEST)
    P_east_bc = DirichletBc(P, grid, Pout, BoundaryLocation.EAST)

    U_west_bc.apply()
    U_east_bc.apply()
    P_west_bc.apply()
    P_east_bc.apply()

    # Models
    Uold = np.copy(U)

    transient = FirstOrderTransientModel(grid, U, Uold, rho, 1.0, dt)
    diffusion = DiffusionModel(grid, U, mu, U_west_bc, U_east_bc)
    advection = advection_class(grid, U, Uhe, rho, 1.0, U_west_bc, U_east_bc)
    pressure = PressureForceModel(grid, P, P_west_bc, P_east_bc)

    advecting = AdvectingVelocityModel(grid, dhat, Uhe, P, U, UU_coeffs)

    mass = MassConservationEquation(
        grid, U, P, dhat, Uhe, rho,
        P_west_bc, P_east_bc,
        U_west_bc, U_east_bc
    )


    # Time loop
    for tStep in range(nTime):
        time += dt
        Uold[:] = U[:]

        print(f"\nncv = {ncv}, scheme = {advection_class.__name__}")

        for i in range(maxIter):
            PP_coeffs.zero()
            PU_coeffs.zero()
            UU_coeffs.zero()
            UP_coeffs.zero()

            # Momentum equation
            UU_coeffs = diffusion.add(UU_coeffs)
            UU_coeffs = advection.add(UU_coeffs)
            UU_coeffs = transient.add(UU_coeffs)
            UP_coeffs = pressure.add(UP_coeffs)


            # Mass equation
            advecting.update()
            PP_coeffs, PU_coeffs = mass.add(PP_coeffs, PU_coeffs)

            # Residuals
            P_resid = PU_coeffs.rP + PP_coeffs.rP
            U_resid = UU_coeffs.rP + UP_coeffs.rP

            PmaxResid = norm(P_resid, np.inf)
            UmaxResid = norm(U_resid, np.inf)

            print(
                "Iteration = {}; Mass max = {:.4e}; Momentum max = {:.4e}".format(
                    i, PmaxResid, UmaxResid
                )
            )

            if PmaxResid < converged and UmaxResid < converged:
                break

            dP, dU = solve(PP_coeffs, PU_coeffs, UP_coeffs, UU_coeffs)

            P[1:-1] += dP
            U[1:-1] += dU

            U_west_bc.apply()
            U_east_bc.apply()
            P_west_bc.apply()
            P_east_bc.apply()

            advecting.update()
            
    # Dynamic head loss coefficient
    Pin = P[0]
    Pout_num = P[-1]

    CD = (Pin - Pout_num) / (0.5 * rho * Uin**2)

    return grid, U, P, Uhe, CD

In [ ]:
ncv_list = [8, 16, 32, 64]

schemes = {
    "UDS": UpwindAdvectionModel,
    "CDS": CDSAdvectionModel,
    "QUICK": QUICKAdvectionModel,
}

results_variable = {}

for scheme_name, scheme_class in schemes.items():
    results_variable[scheme_name] = {}

    for ncv in ncv_list:
        grid, U, P, Uhe, CD = run_variable_duct_case(ncv, scheme_class)

        results_variable[scheme_name][ncv] = {
            "grid": grid,
            "U": U,
            "P": P,
            "Uhe": Uhe,
            "CD": CD,
        }

        print(f"{scheme_name}, ncv = {ncv}, CD = {CD:.6e}")

In [ ]:
print(results_variable.keys())

for key in results_variable.keys():
    print(key, results_variable[key].keys())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

rho = 1000.0
Uin = 2.0
ncv_plot = 64

plt.figure(figsize=(7, 4))

for scheme_name in schemes.keys():
    grid = results_variable[scheme_name][ncv_plot]["grid"]
    U = results_variable[scheme_name][ncv_plot]["U"]

    plt.plot(
        grid.xP,
        U,
        marker="o",
        linestyle="None",
        label=scheme_name
    )



plt.xlabel("x")
plt.ylabel("U")
plt.title(f"Velocity distribution, ncv = {ncv_plot}")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(7, 4))

for scheme_name in schemes.keys():
    grid = results_variable[scheme_name][ncv_plot]["grid"]
    P = results_variable[scheme_name][ncv_plot]["P"]

    plt.plot(
        grid.xP,
        P,
        marker="s",
        linestyle="None",
        label=scheme_name
    )


plt.xlabel("x")
plt.ylabel("P")
plt.title(f"Pressure distribution, ncv = {ncv_plot}")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

for scheme_name in schemes.keys():
    CD_values = [
        abs(results_variable[scheme_name][ncv]["CD"])
        for ncv in ncv_list
    ]

    plt.plot(
        ncv_list,
        CD_values,
        marker="o",
        linestyle="-",
        label=scheme_name
    )

plt.xlabel("ncv")
plt.ylabel("CD")
plt.title("Dynamic head loss convergence")
plt.grid(True, which="both")
plt.legend()
plt.show()

## Discussion
The velocity plot shows that all three schemes give almost the same velocity distribution. The velocity starts from approximately 2 m/s at the inlet, increases through the contraction, reaches a maximum value near the throat, and then decreases back to approximately 2 m/s at the outlet. This behavior is expected from mass conservation. Since the duct area decreases toward the throat, the velocity must increase to maintain a constant mass flow rate. 

The UDS result shows a larger deviation in pressure. In particular, the inlet pressure is higher than the outlet pressure, which indicates an artificial dynamic head loss. This occurs because UDS is first-order accurate and introduces numerical diffusion. In this inviscid variable-area duct problem, that numerical diffusion behaves like an artificial loss mechanism and prevents full pressure recovery. The second-order schemes, CDS and QUICK, give very similar pressure distributions and show better pressure recovery. Their inlet and outlet pressures are close to each other, which is physically expected because the inlet and outlet areas are the same and wall friction is turned off. However, for smaller NCV values, the second-order schemes become unstable and may diverge. This is because the coarse mesh cannot properly resolve the rapid area change near the throat, where the velocity and pressure gradients are large.

The duct has a contraction followed by an expansion, but the inlet and outlet areas are equal. Since wall friction is turned off, there is no physical energy loss. Therefore, the total pressure should remain constant. Since the CD is computed based on the pressure change between inlet and outlet, CD should be zero in this case.

UDS will predict the largest nonzero value of CD at low ncv, because it introduces numerical diffusion. As the mesh is refined from 8 to 64 control volumes, the UDS result should approach CD = 0, but slowly.

CDS and QUICK should give CD than UDS because they are less numerically diffusive. Therefore, their CD values should approach zero faster than UDS. UDS should show approximately first-order convergence, while CDS and QUICK should show approximately second-order convergence.